In [5]:
r = requests.get("http://172.19.0.1:9010/minio/health/live", timeout=3)
print("MinIO →", r.status_code)

MinIO → 200


In [4]:
import requests

for host in ["host.docker.internal", "172.19.0.1"]:
    try:
        r = requests.get(f"http://{host}:8182/catalog/v1/config?warehouse=lakehouse", timeout=3)
        print(host, "→", r.status_code)
    except Exception as e:
        print(host, "→ ERREUR:", type(e).__name__, str(e)[:100])

host.docker.internal → ERREUR: ConnectionError HTTPConnectionPool(host='host.docker.internal', port=8182): Max retries exceeded with url: /catalog/
172.19.0.1 → 200


In [8]:
import shutil, os

# Espace disque
total, used, free = shutil.disk_usage("/tmp")
print(f"Espace /tmp : {free // (1024**2)} Mo libres sur {total // (1024**2)} Mo")

# Permissions
print("Writable /tmp :", os.access("/tmp", os.W_OK))

# Combien de dossiers spark-* déjà présents
import glob
leftovers = glob.glob("/tmp/spark-*")
print(f"Dossiers spark-* existants : {len(leftovers)}")

Espace /tmp : 100 Mo libres sur 79943 Mo
Writable /tmp : True
Dossiers spark-* existants : 4


In [9]:
!rm -rf /tmp/spark-* 2>/dev/null
!mkdir -p /tmp/spark-tmp && chmod 777 /tmp/spark-tmp
print("✅ Nettoyé")

✅ Nettoyé


In [10]:
import os
from datetime import datetime

try:
    from pyspark.sql import SparkSession
    SparkSession.getActiveSession().stop()
except:
    pass

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.5.2,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType

HOST_VM = "172.19.0.1"   # confirmé : Lakekeeper (8182) + MinIO (9010) répondent ici

MINIO_ENDPOINT       = f"http://{HOST_VM}:9010"
MINIO_ACCESS         = "minioadmin"
MINIO_SECRET         = "minioadmin123"
CATALOG_NAME         = "lakehouse"
LAKEKEEPER_URI       = f"http://{HOST_VM}:8182/catalog"
LAKEKEEPER_WAREHOUSE = "lakehouse"

spark = (
    SparkSession.builder
    .appName("SchemaEvolution-Livres")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", LAKEKEEPER_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LAKEKEEPER_WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint", MINIO_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id", MINIO_ACCESS)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", MINIO_SECRET)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.local.dir", "/tmp/spark-tmp")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark connecté — {spark.version}")
print(f"   Catalogue : {LAKEKEEPER_URI}")
print(f"   Storage   : {MINIO_ENDPOINT}")

✅ Spark connecté — 3.4.0
   Catalogue : http://172.19.0.1:8182/catalog
   Storage   : http://172.19.0.1:9010


In [12]:
import requests
r = requests.get("http://172.19.0.1:9010/minio/health/live")
print(r.status_code, r.headers)

200 {'Accept-Ranges': 'bytes', 'Content-Length': '0', 'Server': 'MinIO', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'Vary': 'Origin', 'X-Amz-Id-2': '6288f7c424456b65729155b10570da05022411640ac68a83da601467ee9d5c0a', 'X-Amz-Request-Id': '18C6734A41986C2B', 'X-Content-Type-Options': 'nosniff', 'X-Xss-Protection': '1; mode=block', 'Date': 'Tue, 28 Jul 2026 12:17:45 GMT'}


In [13]:
SILVER_LIVRES = f"{CATALOG_NAME}.silver.livres"

print("Tables silver :", [t.tableName for t in spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.silver").collect()])

schema_avant = [f.name for f in spark.table(SILVER_LIVRES).schema.fields]
print("Schéma AVANT :", schema_avant)

spark.sql(f"""
    ALTER TABLE {SILVER_LIVRES}
    ADD COLUMN format_livre STRING COMMENT 'Poche / Broché / Numérique'
""")

schema_apres = [f.name for f in spark.table(SILVER_LIVRES).schema.fields]
print("✅ Colonne ajoutée :", set(schema_apres) - set(schema_avant))

Tables silver : ['utilisateurs', 'livres', 'commandes', 'stock', 'catalogue', 'activite_temps_reel']
Schéma AVANT : ['auteur', 'livre_id', 'date_publication', 'description', 'disponible', 'editeur', 'genre', 'isbn', 'langue', 'nb_avis', 'nb_pages', 'nb_ventes_total', 'note_moyenne', 'prix_achat', 'prix_vente', 'sous_genre', 'titre', 'updated_at', 'tags_str', '_source', '_ingestion_date', '_couche', 'partition_genre', 'categorie_prix', 'score_popularite', 'marge_pct', 'est_disponible']
✅ Colonne ajoutée : {'format_livre'}


In [14]:
SILVER_LIVRES = f"{CATALOG_NAME}.silver.livres"

print("Tables silver :", [t.tableName for t in spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.silver").collect()])

schema_avant = [f.name for f in spark.table(SILVER_LIVRES).schema.fields]
print("Schéma AVANT :", schema_avant)

spark.sql(f"""
    ALTER TABLE {SILVER_LIVRES}
    ADD COLUMN resume STRING
""")

schema_apres = [f.name for f in spark.table(SILVER_LIVRES).schema.fields]
print("✅ Colonne ajoutée :", set(schema_apres) - set(schema_avant))

Tables silver : ['utilisateurs', 'livres', 'commandes', 'stock', 'catalogue', 'activite_temps_reel']
Schéma AVANT : ['auteur', 'livre_id', 'date_publication', 'description', 'disponible', 'editeur', 'genre', 'isbn', 'langue', 'nb_avis', 'nb_pages', 'nb_ventes_total', 'note_moyenne', 'prix_achat', 'prix_vente', 'sous_genre', 'titre', 'updated_at', 'tags_str', '_source', '_ingestion_date', '_couche', 'partition_genre', 'categorie_prix', 'score_popularite', 'marge_pct', 'est_disponible', 'format_livre']
✅ Colonne ajoutée : {'resume'}
